In [1]:
import os

os.environ["HF_HOME"] = "C:/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "C:/hf_cache/datasets"
os.environ["HF_HUB_CACHE"] = "C:/hf_cache/hub"

import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "../qwen_qlora_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model (no fine-tuning)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Base model loaded")
print("Base model memory footprint (GB):", round(base_model.get_memory_footprint() / (1024**3), 2))

# Load a second copy and attach the fine-tuned LoRA adapter
finetuned_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
finetuned_model = PeftModel.from_pretrained(finetuned_base, ADAPTER_PATH)
print("Fine-tuned model loaded (base + LoRA adapter)")
print("Fine-tuned model memory footprint (GB):", round(finetuned_model.get_memory_footprint() / (1024**3), 2))

C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base model loaded
Base model memory footprint (GB): 1.48
Fine-tuned model loaded (base + LoRA adapter)
Fine-tuned model memory footprint (GB): 1.55


In [3]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# Use the same seed/split logic as training so this is a genuine held-out set
split_dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)
eval_pool = split_dataset["test"]

# Sample a manageable number of examples for Ragas evaluation
# (Ragas evaluation is compute-heavy per example...... so i am keeping this focused)
eval_sample = eval_pool.shuffle(seed=42).select(range(50))

print("Evaluation sample size:", len(eval_sample))
print("\nSample instruction:", eval_sample[0]["instruction"])
print("Sample reference response:", eval_sample[0]["response"])

Evaluation sample size: 50

Sample instruction: edit data on pro account
Sample reference response: Thank you for contacting to us with your request to edit the data on your pro account. We appreciate your dedication to keeping your information up to date and accurate. To assist you further, could you please provide more specific details regarding the data you would like to edit? This will allow us to guide you through the process more efficiently and ensure that your account information reflects your needs accurately.


In [4]:
def generate_response(model, tokenizer, instruction, max_new_tokens=150):
    messages = [{"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic, for fair comparison
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

# Quick test on 3 examples before running the full 50
for i in range(3):
    instruction = eval_sample[i]["instruction"]
    reference = eval_sample[i]["response"]

    base_response = generate_response(base_model, tokenizer, instruction)
    finetuned_response = generate_response(finetuned_model, tokenizer, instruction)

    print(f"--- Example {i+1} ---")
    print("Instruction:", instruction)
    print("\nReference:", reference)
    print("\nBase model:", base_response)
    print("\nFine-tuned model:", finetuned_response)
    print("\n" + "="*80 + "\n")

C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


--- Example 1 ---
Instruction: edit data on pro account

Reference: Thank you for contacting to us with your request to edit the data on your pro account. We appreciate your dedication to keeping your information up to date and accurate. To assist you further, could you please provide more specific details regarding the data you would like to edit? This will allow us to guide you through the process more efficiently and ensure that your account information reflects your needs accurately.

Base model: I'm sorry, but I don't have the capability to edit or manipulate data directly. As an AI language model, my primary function is to provide information and answer questions based on my training data. If you need help with editing data, please let me know what specific task you're trying to accomplish, and I'll do my best to assist you within those parameters.

Fine-tuned model: How exhilarating it is to embark on this transformative journey of editing the data on your esteemed {{Account Typ

In [5]:
base_responses = []
finetuned_responses = []
references = []
instructions = []

for i in range(len(eval_sample)):
    instruction = eval_sample[i]["instruction"]
    reference = eval_sample[i]["response"]

    base_resp = generate_response(base_model, tokenizer, instruction)
    finetuned_resp = generate_response(finetuned_model, tokenizer, instruction)

    instructions.append(instruction)
    references.append(reference)
    base_responses.append(base_resp)
    finetuned_responses.append(finetuned_resp)

    if (i + 1) % 10 == 0:
        print(f"Completed {i + 1}/{len(eval_sample)}")

print("\nAll responses generated")

Completed 10/50
Completed 20/50
Completed 30/50
Completed 40/50
Completed 50/50

All responses generated


In [6]:
import gc

del base_model
del finetuned_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed")

GPU memory freed


In [7]:
import json

eval_data = {
    "instructions": instructions,
    "references": references,
    "base_responses": base_responses,
    "finetuned_responses": finetuned_responses,
}

with open("../eval_responses.json", "w", encoding="utf-8") as f:
    json.dump(eval_data, f, indent=2)

print("Saved eval responses to ../eval_responses.json")

Saved eval responses to ../eval_responses.json
